In [422]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass

In [423]:
@dataclass
class Config:
    vocab_size : int = 50_000
    max_seq_len : int = 1024
    n_layers : int = 12
    n_heads : int = 12
    d_model : int = 768
    dropout : float = 0.1
    pad_token_id : int = 49999
    use_sinusoidal_pos_emb : bool = False

    @property
    def d_head(self):
        assert self.d_model % self.n_heads == 0, ("d_model must be divisible by n_heads")
        return self.d_model // self.n_heads

In [424]:
class AbsolutePositionalEmbedding(nn.Module):
    def __init__(self, config:Config):
        super().__init__()
        self.token_embeddings = nn.Embedding(num_embeddings=config.vocab_size, embedding_dim= config.d_model)
        self.position_embeddings = nn.Embedding(num_embeddings = config.max_seq_len, embedding_dim= config.d_model)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, input_ids):
        batch_size, seq_len = input_ids.shape
        token_embs = self.token_embeddings(input_ids)
        positions = torch.arange(seq_len, device=input_ids.device)
        position_embs = self.position_embeddings(positions)
        x = token_embs + position_embs
        x = self.dropout(x)
        return x

In [425]:
class SinusoidalPositions(nn.Module):
    def __init__(self, config:Config):
        super().__init__()

        position = torch.arange(config.max_seq_len).unsqueeze(1)

        # wk = 1/10000 ^ 2k/d (Refer : https://kazemnejad.com/blog/transformer_architecture_positional_encoding/)
        # apply log first and then apply exp which mean exp and log cancels out
        
        div_term = torch.exp(
            torch.arange(0, config.d_model, 2) * (-math.log(10000.0) / config.d_model)
        )

        spe = torch.zeros(1, config.max_seq_len, config.d_model)
        spe[0, :, 0::2] = torch.sin(position * div_term)
        spe[0, :, 1::2] = torch.cos(position * div_term[: spe[0, :, 1::2].shape[1]])

        self.register_buffer("spe", spe)

    def forward(self, seq_len: int):
        return self.spe[:, :seq_len, :]
        
    

In [426]:
class SinusoidalPositionalEmbedding(nn.Module):
    def __init__(self, config:Config):
        super().__init__()
        self.token_embeddings = nn.Embedding(num_embeddings=config.vocab_size, embedding_dim=config.d_model)
        self.position_embeddings = SinusoidalPositions(config)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, inputs):
        batch_size, seq_len = inputs.shape
        token_embs = self.token_embeddings(inputs)
        position_embs = self.position_embeddings(seq_len)
        x  = token_embs + position_embs
        x = self.dropout(x)
        return x

In [427]:
class CausalSelfAttention(nn.Module):
    def __init__(self, config: Config):
        super().__init__()

        self.d_model = config.d_model
        self.max_seq_len = config.max_seq_len
        self.n_heads = config.n_heads
        self.dropout = config.dropout
        self.d_model = config.d_model
        self.d_head = config.d_head

        self.q_proj = nn.Linear(self.d_model, self.d_model)
        self.k_proj = nn.Linear(self.d_model, self.d_model)
        self.v_proj = nn.Linear(self.d_model, self.d_model)
        self.o_proj = nn.Linear(self.d_model, self.d_model)

        self.attn_dropout = nn.Dropout(self.dropout)
        self.resid_dropout = nn.Dropout(self.dropout)

        causal_mask = torch.tril(
            torch.ones(
                config.max_seq_len,
                config.max_seq_len,
                dtype=torch.bool,
            )
        )

        causal_mask = causal_mask.view(
            1, 1, config.max_seq_len, config.max_seq_len
        )

        self.register_buffer("causal_mask", causal_mask)

    def forward(self, x, attention_mask=None, return_attn=False):
        """
        x:              [batch_size, seq_len, d_model]
        attention_mask: [batch_size, seq_len]

        returns:
            out: [batch_size, seq_len, d_model]
        """

        batch_size, seq_len, d_model = x.shape

        assert d_model == self.d_model
        assert seq_len <= self.max_seq_len

        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        # [B, T, D] -> [B, T, H, d_head]
        # q = q.view(batch_size, seq_len, self.n_heads, self.d_head)
        # k = k.view(batch_size, seq_len, self.n_heads, self.d_head)
        # v = v.view(batch_size, seq_len, self.n_heads, self.d_head) same as

        q = q.reshape(batch_size, seq_len, self.n_heads, self.d_head)
        k = q.reshape(batch_size, seq_len, self.n_heads, self.d_head)
        v = v.reshape(batch_size, seq_len, self.n_heads, self.d_head)

        # [B, T, H, d_head] -> [B, H, T, d_head]
        q = q.transpose(1, 2)
        k = k.transpose(1, 2)
        v = v.transpose(1, 2)

        scores = torch.matmul(q, k.transpose(-2, -1)) # or scores = q @ k.transpose(-2, -1)
        # [B, H, T, T]

        scores = scores / math.sqrt(self.d_head)

        causal_mask = self.causal_mask[:, :, :seq_len, :seq_len]
        # [1, 1, T, T]

        if attention_mask is not None:
            key_padding_mask = attention_mask[:, None, None, :].bool()
            # [B, 1, 1, T]

            combined_mask = causal_mask & key_padding_mask
            # [B, 1, T, T]
        else:
            combined_mask = causal_mask
            # [1, 1, T, T]

        scores = scores.masked_fill(~combined_mask, float("-inf"))

        attn_weights = torch.softmax(scores, dim=-1)
        # [B, H, T, T]

        attn_weights = self.attn_dropout(attn_weights)

        out = torch.matmul(attn_weights, v) # or out = attn_weights @ v [B, H, T, T] * [B, H, T, d_head]
        # [B, H, T, d_head]

        out = out.transpose(1, 2)
        # [B, T, H, d_head]

        #out = out.contiguous().view(batch_size, seq_len, d_model) same as
        out = out.reshape(batch_size, seq_len, d_model)
        # [B, T, D]

        out = self.o_proj(out)
        out = self.resid_dropout(out)

        if return_attn:
            return out, attn_weights, combined_mask

        return out

In [428]:
# Inputs  Creation just using torch input functions
batch_size = 4
pad_token_id = 0
config = Config(25, 10, 4, 4, 256, 0.1)
input_ids = torch.randint(1, config.vocab_size, (batch_size, config.max_seq_len))
input_lengths = torch.randint(1, config.max_seq_len + 1, (batch_size,))
attention_mask = (torch.arange(config.max_seq_len).unsqueeze(0) < input_lengths.unsqueeze(1)).long()
print(attention_mask)
input_ids = input_ids.masked_fill(attention_mask == 0, pad_token_id)
print(input_ids)
x = AbsolutePositionalEmbedding(config)(input_ids)
print(x)
x = SinusoidalPositionalEmbedding(config)(input_ids)
print(x)
out = CausalSelfAttention(config)(x)
print(out)




                              

tensor([[1, 1, 1, 1, 1, 1, 1, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 0, 0, 0],
        [1, 1, 1, 1, 0, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 0, 0, 0]])
tensor([[24, 13, 12,  9, 14,  8,  5,  0,  0,  0],
        [24,  6, 13, 23, 13,  5, 13,  0,  0,  0],
        [19, 23, 17,  3,  0,  0,  0,  0,  0,  0],
        [10, 11, 17, 17, 21,  9, 18,  0,  0,  0]])
tensor([[[-1.4945, -1.1589, -1.8558,  ...,  1.0663,  0.6257,  0.0000],
         [-3.9288,  2.1966,  0.0201,  ..., -0.0000, -0.6221,  2.6510],
         [ 0.1492, -0.8051,  0.6529,  ..., -0.0000,  0.0000, -2.2927],
         ...,
         [-0.7111, -1.8425,  0.6197,  ..., -1.7341, -0.6233,  1.5245],
         [ 0.0000, -0.3671, -0.3980,  ...,  1.2141,  1.6273,  1.2100],
         [ 0.0634, -2.8957,  2.0294,  ...,  1.1119,  0.5527,  1.0640]],

        [[-1.4945, -1.1589, -1.8558,  ...,  1.0663,  0.6257,  0.7437],
         [ 0.3446,  0.2219,  3.0416,  ..., -0.7774, -0.9155, -0.1219],
         [-3.1114,  1.4205, -1.3453,  ..., -1.3593,  1

In [429]:
# Relative Positional Embedding

In [430]:
class TokenEmbeddingLayer(nn.Module):
    def __init__(self, config: Config):
        super().__init__()

        self.token_embeddings = nn.Embedding(
            num_embeddings=config.vocab_size,
            embedding_dim=config.d_model
        )

        self.dropout = nn.Dropout(config.dropout)

    def forward(self, input_ids):
        """
        input_ids: [batch_size, seq_len]

        returns:
            x: [batch_size, seq_len, d_model]
        """

        x = self.token_embeddings(input_ids)
        x = self.dropout(x)

        return x

In [431]:
class RelativePositionBias(nn.Module):
    def __init__(self, config: Config, max_distance: int | None = None):
        super().__init__()

        self.n_heads = config.n_heads
        self.max_distance = max_distance or config.max_seq_len - 1

        # One learned bias per relative distance, per attention head.
        #
        # distance 0 means current token
        # distance 1 means one token back
        # distance 2 means two tokens back
        # ...
        self.relative_bias = nn.Embedding(
            num_embeddings=self.max_distance + 1,
            embedding_dim=config.n_heads, # very important 
        )

    def forward(self, seq_len: int, device):
        """
        returns:
            bias: [1, n_heads, seq_len, seq_len]
        """

        query_positions = torch.arange(seq_len, device=device)[:, None] # or  torch.arange(seq_len, device=device).unsqueeze(1)
        # [seq_len, 1]

        key_positions = torch.arange(seq_len, device=device)[None, :] # or  torch.arange(seq_len, device=device).unsqueeze(0)
        # [1, seq_len]

        relative_distance = query_positions - key_positions
        # [seq_len, seq_len]

        relative_distance = relative_distance.clamp(
            min=0,
            max=self.max_distance,
        )
        # [seq_len, seq_len]

        bias = self.relative_bias(relative_distance)
        # [seq_len, seq_len, n_heads]

        bias = bias.permute(2, 0, 1)
        # [n_heads, seq_len, seq_len]

        bias = bias.unsqueeze(0)
        # [1, n_heads, seq_len, seq_len]

        return bias

In [432]:
class RelativeCausalSelfAttention(nn.Module):
    def __init__(self, config: Config):
        super().__init__()

        self.d_model = config.d_model
        self.max_seq_len = config.max_seq_len
        self.n_heads = config.n_heads
        self.dropout = config.dropout
        self.d_head = config.d_head

        self.q_proj = nn.Linear(self.d_model, self.d_model)
        self.k_proj = nn.Linear(self.d_model, self.d_model)
        self.v_proj = nn.Linear(self.d_model, self.d_model)
        self.o_proj = nn.Linear(self.d_model, self.d_model)

        self.relative_position_bias = RelativePositionBias(config)

        self.attn_dropout = nn.Dropout(self.dropout)
        self.resid_dropout = nn.Dropout(self.dropout)

        causal_mask = torch.tril(
            torch.ones(
                config.max_seq_len,
                config.max_seq_len,
                dtype=torch.bool,
            )
        )

        # causal_mask = causal_mask.view(
        #     1, 1, config.max_seq_len, config.max_seq_len
        # ) same as
        causal_mask = causal_mask[None, None, :, :]

        self.register_buffer("causal_mask", causal_mask)

    def forward(self, x, attention_mask=None, return_attn=False):
        """
        x:              [batch_size, seq_len, d_model]
        attention_mask: [batch_size, seq_len]

        returns:
            out: [batch_size, seq_len, d_model]
        """

        batch_size, seq_len, d_model = x.shape

        assert d_model == self.d_model
        assert seq_len <= self.max_seq_len

        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        # [B, T, D] -> [B, T, H, d_head]
        # q = q.view(batch_size, seq_len, self.n_heads, self.d_head)
        # k = k.view(batch_size, seq_len, self.n_heads, self.d_head)
        # v = v.view(batch_size, seq_len, self.n_heads, self.d_head) same as

        q = q.reshape(batch_size, seq_len, self.n_heads, self.d_head)
        k = q.reshape(batch_size, seq_len, self.n_heads, self.d_head)
        v = v.reshape(batch_size, seq_len, self.n_heads, self.d_head)

        # [B, T, H, d_head] -> [B, H, T, d_head]
        q = q.transpose(1, 2)
        k = k.transpose(1, 2)
        v = v.transpose(1, 2)

        scores = q @ k.transpose(-2, -1)
        # [B, H, T, T]

        scores = scores / math.sqrt(self.d_head)

        # Add relative position bias here
        rel_bias = self.relative_position_bias(
            seq_len=seq_len,
            device=x.device,
        )
        # [1, H, T, T]

        scores = scores + rel_bias
        # [B, H, T, T]

        causal_mask = self.causal_mask[:, :, :seq_len, :seq_len]
        # [1, 1, T, T]

        if attention_mask is not None:
            key_padding_mask = attention_mask[:, None, None, :].bool()
            # [B, 1, 1, T]

            combined_mask = causal_mask & key_padding_mask
            # [B, 1, T, T]
        else:
            combined_mask = causal_mask
            # [1, 1, T, T]

        scores = scores.masked_fill(~combined_mask, float("-inf"))

        attn_weights = torch.softmax(scores, dim=-1)
        # [B, H, T, T]

        attn_weights = self.attn_dropout(attn_weights)

        out = attn_weights @ v
        # [B, H, T, d_head]

        out = out.transpose(1, 2)
        # [B, T, H, d_head]

        #out = out.contiguous().view(batch_size, seq_len, d_model) same as
        out = out.reshape(batch_size, seq_len, d_model)
        # [B, T, D]

        out = self.o_proj(out)
        out = self.resid_dropout(out)

        if return_attn:
            return out, attn_weights, combined_mask, rel_bias

        return out

In [433]:
batch_size = 3
max_seq_len = 6
vocab_size = 100

lengths = torch.randint(
    low=1,
    high=max_seq_len + 1,
    size=(batch_size,)
)

input_ids = torch.randint(
    low=1,
    high=vocab_size,
    size=(batch_size, max_seq_len)
)

attention_mask = (
    torch.arange(max_seq_len).unsqueeze(0) < lengths.unsqueeze(1)
).long()

input_ids = input_ids.masked_fill(attention_mask == 0, pad_token_id)

config = Config(
    vocab_size=vocab_size,
    max_seq_len=max_seq_len,
    d_model=8,
    n_heads=2,
    n_layers=2,
    dropout=0.0
)

embedding_layer = TokenEmbeddingLayer(config)
attention_layer = RelativeCausalSelfAttention(config)

x = embedding_layer(input_ids)

out, attn_weights, combined_mask, rel_bias = attention_layer(
    x,
    attention_mask=attention_mask,
    return_attn=True,
)

print("lengths:")
print(lengths)

print("\ninput_ids:")
print(input_ids)

print("\nattention_mask:")
print(attention_mask)

print("\nx shape:", x.shape)
print("relative bias shape:", rel_bias.shape)
print("attention weights shape:", attn_weights.shape)
print("combined mask shape:", combined_mask.shape)
print("out shape:", out.shape)

lengths:
tensor([4, 2, 5])

input_ids:
tensor([[94, 75,  2, 79,  0,  0],
        [42, 46,  0,  0,  0,  0],
        [41, 95, 84, 35, 29,  0]])

attention_mask:
tensor([[1, 1, 1, 1, 0, 0],
        [1, 1, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 0]])

x shape: torch.Size([3, 6, 8])
relative bias shape: torch.Size([1, 2, 6, 6])
attention weights shape: torch.Size([3, 2, 6, 6])
combined mask shape: torch.Size([3, 1, 6, 6])
out shape: torch.Size([3, 6, 8])


In [434]:
# ROPE - Rotational Positional Embedding

In [435]:
class TokenEmbeddingLayer(nn.Module):
    def __init__(self, config):
        super().__init__()

        self.token_embeddings = nn.Embedding(
            num_embeddings=config.vocab_size,
            embedding_dim=config.d_model,
            padding_idx=config.pad_token_id,
        )

        self.dropout = nn.Dropout(config.dropout)

    def forward(self, input_ids):
        """
        input_ids: [batch_size, seq_len]

        returns:
            x: [batch_size, seq_len, d_model]
        """

        x = self.token_embeddings(input_ids)
        x = self.dropout(x)

        return x

In [436]:
class RotaryPositionalEmbedding(nn.Module):
    def __init__(self, config, base: float = 10000.0):
        super().__init__()

        assert config.d_head % 2 == 0, "d_head must be even for RoPE"

        self.d_head = config.d_head
        self.max_seq_len = config.max_seq_len
        # RoPE is applied per attention head ([B, H, T, d_head]) , so the frequency dimension should be based on config.d_head
        inv_freq = torch.exp(
            torch.arange(0, config.d_head, 2, dtype=torch.float32)
            * (-math.log(base) / config.d_head)
        )
        # [d_head / 2]


        positions = torch.arange(config.max_seq_len, dtype=torch.float32).unsqueeze(1)
        # [1 * max_seq_len]

        angles = positions * inv_freq
        # [max_seq_len, d_head / 2]

        cos = torch.cos(angles)
        sin = torch.sin(angles)

        # Make shapes broadcastable with q/k:
        # q/k: [B, H, T, d_head]
        # cos/sin: [1, 1, T, d_head / 2]
        cos = cos[None, None, :, :]
        sin = sin[None, None, :, :]

        self.register_buffer("cos", cos)
        self.register_buffer("sin", sin)

    def forward(self, x):
        """
        x: [batch_size, n_heads, seq_len, d_head]

        returns:
            x_rotated: [batch_size, n_heads, seq_len, d_head]
        """

        batch_size, n_heads, seq_len, d_head = x.shape

        assert d_head == self.d_head
        assert seq_len <= self.max_seq_len

        cos = self.cos[:, :, :seq_len, :]
        sin = self.sin[:, :, :seq_len, :]

        x_even = x[..., 0::2]
        x_odd = x[..., 1::2]

        # [x_rotated_even     = [cos_theta, -sin_theta   [x_even, 
        #  x_rotated_odd]        sin_theat, cos_theta ]    x_odd]
        
        x_rotated_even = x_even * cos - x_odd * sin
        x_rotated_odd = x_even * sin + x_odd * cos

        x_rotated = torch.empty_like(x)
        x_rotated[..., 0::2] = x_rotated_even
        x_rotated[..., 1::2] = x_rotated_odd

        return x_rotated

In [437]:
class RoPECausalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()

        self.d_model = config.d_model
        self.max_seq_len = config.max_seq_len
        self.n_heads = config.n_heads
        self.dropout = config.dropout
        self.d_head = config.d_head

        self.q_proj = nn.Linear(self.d_model, self.d_model)
        self.k_proj = nn.Linear(self.d_model, self.d_model)
        self.v_proj = nn.Linear(self.d_model, self.d_model)
        self.o_proj = nn.Linear(self.d_model, self.d_model)

        self.rope = RotaryPositionalEmbedding(config)

        self.attn_dropout = nn.Dropout(self.dropout)
        self.resid_dropout = nn.Dropout(self.dropout)

        causal_mask = torch.tril(
            torch.ones(
                config.max_seq_len,
                config.max_seq_len,
                dtype=torch.bool,
            )
        )

       # causal_mask = causal_mask.view(
       #     1, 1, config.max_seq_len, config.max_seq_len
       # ) same as

        causal_mask = causal_mask[None, None, :, :]

        self.register_buffer("causal_mask", causal_mask)

    def forward(self, x, attention_mask=None, return_attn=False):
        """
        x:              [batch_size, seq_len, d_model]
        attention_mask: [batch_size, seq_len]

        returns:
            out: [batch_size, seq_len, d_model]
        """

        batch_size, seq_len, d_model = x.shape

        assert d_model == self.d_model
        assert seq_len <= self.max_seq_len

        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        # [B, T, D] -> [B, T, H, d_head]
        # q = q.view(batch_size, seq_len, self.n_heads, self.d_head)
        # k = k.view(batch_size, seq_len, self.n_heads, self.d_head)
        # v = v.view(batch_size, seq_len, self.n_heads, self.d_head) same as

        q = q.reshape(batch_size, seq_len, self.n_heads, self.d_head)
        k = q.reshape(batch_size, seq_len, self.n_heads, self.d_head)
        v = v.reshape(batch_size, seq_len, self.n_heads, self.d_head)

        # [B, T, H, d_head] -> [B, H, T, d_head]
        q = q.transpose(1, 2)
        k = k.transpose(1, 2)
        v = v.transpose(1, 2)

        # Apply RoPE to Q and K only
        q = self.rope(q)
        k = self.rope(k)

        scores = q @ k.transpose(-2, -1)
        # [B, H, T, T]

        scores = scores / math.sqrt(self.d_head)

        causal_mask = self.causal_mask[:, :, :seq_len, :seq_len]
        # [1, 1, T, T]

        if attention_mask is not None:
            key_padding_mask = attention_mask[:, None, None, :].bool()
            # [B, 1, 1, T]

            combined_mask = causal_mask & key_padding_mask
            # [B, 1, T, T]
        else:
            combined_mask = causal_mask
            # [1, 1, T, T]

        scores = scores.masked_fill(
            ~combined_mask,
            torch.finfo(scores.dtype).min,
        )

        attn_weights = torch.softmax(scores, dim=-1)
        # [B, H, T, T]

        attn_weights = self.attn_dropout(attn_weights)

        out = attn_weights @ v
        # [B, H, T, d_head]

        out = out.transpose(1, 2)
        # [B, T, H, d_head]

        #out = out.contiguous().view(batch_size, seq_len, d_model) same as 
        out = out.reshape(batch_size, seq_len, d_model)
        # [B, T, D]

        out = self.o_proj(out)
        out = self.resid_dropout(out)
        
        if return_attn:
            return out, attn_weights, combined_mask

        return out

In [438]:
class FeedForward(nn.Module):
    def __init__(self, config: Config):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(config.d_model, 4 * config.d_model),
            nn.GELU(), # GELU (Gaussian Error Linear Unit : x * Phi(x) Where (Phi(x)) is the Cumulative Distribution Function (CDF) of the standard normal distribution [1]
            nn.Linear(4 * config.d_model, config.d_model),
            nn.Dropout(config.dropout),
        )

    def forward(self, x):
        return self.net(x)

In [439]:
class DecoderBlock(nn.Module):
    def __init__(self, config: Config):
        super().__init__()

        self.ln_1 = nn.LayerNorm(config.d_model)
        self.attn = CausalSelfAttention(config)

        self.ln_2 = nn.LayerNorm(config.d_model)
        self.mlp = FeedForward(config)

    def forward(self, x, attention_mask=None):
        """
        x: [batch_size, seq_len, d_model]
        """

        # Attention sub-layer
        x = x + self.attn(
            self.ln_1(x),
            attention_mask=attention_mask,
        )

        # MLP sub-layer
        x = x + self.mlp(self.ln_2(x))

        # Optional cleanup for padded query positions
        if attention_mask is not None:
            x = x * attention_mask[:, :, None].to(x.dtype)

        return x

In [440]:
class GPTDecoder(nn.Module):
    def __init__(self, config: Config):
        super().__init__()

        self.config = config

        if config.use_sinusoidal_pos_emb:
            self.embedding = SinusoidalPositionalEmbedding(config)
        else:
            self.embedding = AbsolutePositionalEmbedding(config)

        self.blocks = nn.ModuleList([
            DecoderBlock(config)
            for _ in range(config.n_layers)
        ])

        self.final_ln = nn.LayerNorm(config.d_model)

        self.lm_head = nn.Linear(
            config.d_model,
            config.vocab_size,
            bias=False,
        )

        # Optional weight tying:
        # output projection shares weights with token embedding. The shared weight is updated once using the accumulated gradient.
        self.lm_head.weight = self.embedding.token_embeddings.weight

    def forward(self, input_ids, attention_mask=None, labels=None):
        """
        input_ids:      [batch_size, seq_len]
        attention_mask: [batch_size, seq_len]
        labels:         [batch_size, seq_len]

        returns:
            logits: [batch_size, seq_len, vocab_size]
            loss: scalar if labels are provided
        """

        batch_size, seq_len = input_ids.shape

        assert seq_len <= self.config.max_seq_len

        x = self.embedding(input_ids)
        # [B, T, D]

        for block in self.blocks:
            x = block(
                x,
                attention_mask=attention_mask,
            )

        x = self.final_ln(x)
        # [B, T, D]

        logits = self.lm_head(x)
        # [B, T, vocab_size]

        loss = None

        # This is the most important conceptual jump in decoder-only LMs. 
        # The model predicts a next-token distribution at every position in parallel during training, 
        # but during generation we usually use only the last position’s logits

        # Lets assume input_ids = [10, 20, 30, 40]
        # x shape = [seq_len, d_model] = [4, d_model]
        # So there is one vector per input token:
        # x_0 for token 10
        # x_1 for token 20
        # x_2 for token 30
        # x_3 for token 40

        # After all decoder blocks, we still have one hidden vector per position:
        # h_0, h_1, h_2, h_3  hidden_states: [4, d_model]
        # Then the LM head is applied to every position:
        # logits = lm_head(hidden_states)
        # logits: [4, vocab_size]

        # logits[0] → prediction distribution after seeing token 10
        # logits[1] → prediction distribution after seeing tokens 10, 20
        # logits[2] → prediction distribution after seeing tokens 10, 20, 30
        # logits[3] → prediction distribution after seeing tokens 10, 20, 30, 40
        
        # That means the model predicts next-token logits at every position in one forward pass.

        
        if labels is not None:
            shift_logits = logits[:, :-1, :] # dropping beacause we dont predict the next token for the last token 
            shift_labels = labels[:, 1:]

            loss = F.cross_entropy(
                shift_logits.reshape(-1, self.config.vocab_size), # -1 in reshape i dont care what is the fist dimension (flatten it) but second dimension should be vocab_size
                shift_labels.reshape(-1),  
                ignore_index=-100,
            )
        return {
            "logits": logits,
            "loss": loss,
        }

    @torch.no_grad()
    def generate(
        self,
        input_ids,
        max_new_tokens: int,
        temperature: float = 1.0,
        top_k: int | None = None,
        top_p: float | None = None,
        eos_token_id: int | None = None,
    ):
        """
        Autoregressive generation.
    
        input_ids: [batch_size, prompt_len]
    
        returns:
            generated_ids: [batch_size, prompt_len + max_new_tokens]
        """
    
        self.eval()
    
        for _ in range(max_new_tokens):
            # Keep only the latest max_seq_len tokens.
            input_ids_cond = input_ids[:, -self.config.max_seq_len:]
    
            # For this simple setup, assume pad_token_id marks padding.
            attention_mask = (input_ids_cond != self.config.pad_token_id).long()
    
            outputs = self(
                input_ids=input_ids_cond,
                attention_mask=attention_mask,
                labels=None,
            )
    
            logits = outputs["logits"]
            # [batch_size, seq_len, vocab_size]
    
            # Use only the final position to predict the next token.
            next_token_logits = logits[:, -1, :]
            # [batch_size, vocab_size]
    
            if temperature == 0:
                # Greedy decoding
                next_token = torch.argmax(
                    next_token_logits,
                    dim=-1,
                    keepdim=True,
                )
    
            else:
                # Temperature scaling
                next_token_logits = next_token_logits / temperature
    
                # -------------------------
                # Top-k filtering
                # -------------------------
                if top_k is not None:
                    top_k = min(top_k, next_token_logits.size(-1))
    
                    values, _ = torch.topk(
                        next_token_logits,
                        k=top_k,
                        dim=-1,
                    )
                    # values: [batch_size, top_k] return both values and indices or values 
    
                    min_values = values[:, -1].unsqueeze(-1)
                    # [batch_size, 1]

                    # torch.where(condition, input_if_true, input_if_false)
                    next_token_logits = torch.where(
                        next_token_logits < min_values, # condition
                        torch.full_like(next_token_logits, float("-inf")), # input_if_true
                        next_token_logits, # input_if_false
                    ) 
    
                # -------------------------
                # Top-p / nucleus filtering
                # -------------------------
                if top_p is not None:
                    assert 0.0 < top_p <= 1.0, "top_p must be in (0, 1]"
    
                    sorted_logits, sorted_indices = torch.sort(
                        next_token_logits,
                        descending=True,
                        dim=-1,
                    )
                    # sorted_logits:  [batch_size, vocab_size]
                    # sorted_indices: [batch_size, vocab_size]
    
                    sorted_probs = torch.softmax(sorted_logits, dim=-1)
                    # [batch_size, vocab_size]
    
                    cumulative_probs = torch.cumsum(sorted_probs, dim=-1)
                    # [batch_size, vocab_size]
    
                    # Remove tokens after cumulative probability exceeds top_p.
                    sorted_indices_to_remove = cumulative_probs > top_p
    
                    # Shift mask right so we keep the first token that crosses top_p.
                    sorted_indices_to_remove[:, 1:] = (
                        sorted_indices_to_remove[:, :-1].clone()
                    )
                    sorted_indices_to_remove[:, 0] = False
    
                    # Map the sorted mask back to original vocab indices.
                    indices_to_remove = torch.zeros_like(
                        next_token_logits,
                        dtype=torch.bool,
                    )
    
                    indices_to_remove.scatter_(
                        dim=-1,
                        index=sorted_indices,
                        src=sorted_indices_to_remove,
                    )
    
                    next_token_logits = next_token_logits.masked_fill(
                        indices_to_remove,
                        float("-inf"),
                    )
    
                probs = torch.softmax(next_token_logits, dim=-1)
                # [batch_size, vocab_size]

                # A multinomial probability distribution is a generalization of the binomial distribution.
                # While a binomial distribution models events with only two possible outcomes (like flipping a coin: heads or tails), 
                # a multinomial distribution models events with three or more distinct outcomes (like rolling a six-sided die, or picking the next word in an LLM vocabulary).
                # It calculates the probability of a specific combination of outcomes occurring across a fixed number of independent trials.
    
                next_token = torch.multinomial(
                    probs,
                    num_samples=1,
                )
                # [batch_size, 1]
    
            input_ids = torch.cat(
                [input_ids, next_token],
                dim=1,
            )
    
            if eos_token_id is not None:
                if torch.all(next_token.squeeze(-1) == eos_token_id):
                    break
    
        return input_ids

In [441]:
# -------------------------
# Create random variable-length input_ids and masks
# -------------------------

torch.manual_seed(42)

batch_size = 3
max_seq_len = 6
vocab_size = 100
pad_token_id = 0

lengths = torch.randint(
    low=1,
    high=max_seq_len + 1,
    size=(batch_size,)
)

input_ids = torch.randint(
    low=1,
    high=vocab_size,
    size=(batch_size, max_seq_len)
)

attention_mask = (
    torch.arange(max_seq_len).unsqueeze(0) < lengths.unsqueeze(1)
).long()

input_ids = input_ids.masked_fill(
    attention_mask == 0,
    pad_token_id,
)

labels = input_ids.clone()
labels = labels.masked_fill(attention_mask == 0, -100)

print("lengths:")
print(lengths)

print("\ninput_ids:")
print(input_ids)

print("\nattention_mask:")
print(attention_mask)

print("\nlabels before shift:")
print(labels)

print("\nshift_labels used in loss:")
print(labels[:, 1:])


# -------------------------
# Run full model
# -------------------------

config = Config(
    vocab_size=vocab_size,
    max_seq_len=max_seq_len,
    d_model=32,
    n_heads=4,
    n_layers=2,
    dropout=0.0,
    pad_token_id=pad_token_id,
)

model = GPTDecoder(config)

outputs = model(
    input_ids=input_ids,
    attention_mask=attention_mask,
    labels=labels,
)

logits = outputs["logits"]
loss = outputs["loss"]

print("\nlogits shape:")
print(logits.shape)

print("\nloss:")
print(loss)


# -------------------------
# Verify backward pass
# -------------------------

loss.backward()

print("\nGradient exists for token embedding weight?")
print(model.embedding.token_embeddings.weight.grad is not None)

print("\nGradient shape:")
print(model.embedding.token_embeddings.weight.grad.shape)

lengths:
tensor([1, 6, 5])

input_ids:
tensor([[59,  0,  0,  0,  0,  0],
        [ 3, 68, 77, 81, 82, 91],
        [23, 93, 59, 40, 32,  0]])

attention_mask:
tensor([[1, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 0]])

labels before shift:
tensor([[  59, -100, -100, -100, -100, -100],
        [   3,   68,   77,   81,   82,   91],
        [  23,   93,   59,   40,   32, -100]])

shift_labels used in loss:
tensor([[-100, -100, -100, -100, -100],
        [  68,   77,   81,   82,   91],
        [  93,   59,   40,   32, -100]])

logits shape:
torch.Size([3, 6, 100])

loss:
tensor(21.9101, grad_fn=<NllLossBackward0>)

Gradient exists for token embedding weight?
True

Gradient shape:
torch.Size([100, 32])


In [442]:
# -------------------------
# Create random variable-length input_ids and masks
# -------------------------

torch.manual_seed(42)

batch_size = 3
max_seq_len = 6
vocab_size = 100
pad_token_id = 0

lengths = torch.randint(
    low=1,
    high=max_seq_len + 1,
    size=(batch_size,)
)

input_ids = torch.randint(
    low=1,
    high=vocab_size,
    size=(batch_size, max_seq_len)
)

attention_mask = (
    torch.arange(max_seq_len).unsqueeze(0) < lengths.unsqueeze(1)
).long()

input_ids = input_ids.masked_fill(
    attention_mask == 0,
    pad_token_id,
)

labels = input_ids.clone()
labels = labels.masked_fill(attention_mask == 0, -100)

print("lengths:")
print(lengths)

print("\ninput_ids:")
print(input_ids)

print("\nattention_mask:")
print(attention_mask)

print("\nlabels before shift:")
print(labels)

print("\nshift_labels used in loss:")
print(labels[:, 1:])


# -------------------------
# Run full model
# -------------------------

config = Config(
    vocab_size=vocab_size,
    max_seq_len=max_seq_len,
    d_model=32,
    n_heads=4,
    n_layers=2,
    dropout=0.0,
    pad_token_id=pad_token_id,
)

model = GPTDecoder(config)

outputs = model(
    input_ids=input_ids,
    attention_mask=attention_mask,
    labels=labels,
)

logits = outputs["logits"]
loss = outputs["loss"]

print("\nlogits shape:")
print(logits.shape)

print("\nloss:")
print(loss)


# -------------------------
# Verify backward pass
# -------------------------

loss.backward()

print("\nGradient exists for token embedding weight?")
print(model.embedding.token_embeddings.weight.grad is not None)

print("\nGradient shape:")
print(model.embedding.token_embeddings.weight.grad.shape)


# -------------------------
# Generation test: single prompt
# -------------------------

# Use only real tokens from the first sequence.
# Do NOT pass padded tokens into generation for this simple test.
prompt_len = int(attention_mask[0].sum().item())
prompt = input_ids[0:1, :prompt_len]

print("\nPrompt used for generation:")
print(prompt)

generated = model.generate(
    input_ids=prompt,
    max_new_tokens=5,
    temperature=1.0,
    top_k=10,
    top_p=0.9,
    eos_token_id=None,
)

print("\nGenerated token IDs with top-k + top-p sampling:")
print(generated)

print("\nGenerated shape:")
print(generated.shape)


# -------------------------
# Greedy generation test
# -------------------------

generated_greedy = model.generate(
    input_ids=prompt,
    max_new_tokens=5,
    temperature=0,
    top_k=None,
    top_p=None,
    eos_token_id=None,
)

print("\nGenerated token IDs with greedy decoding:")
print(generated_greedy)

print("\nGreedy generated shape:")
print(generated_greedy.shape)


# -------------------------
# Optional: batch generation test
# -------------------------

# For a simple batch-generation test, use the first token from each sequence.
# This avoids dealing with variable-length prompts during generation.
batch_prompt = input_ids[:, :1]

print("\nBatch prompt:")
print(batch_prompt)

generated_batch = model.generate(
    input_ids=batch_prompt,
    max_new_tokens=5,
    temperature=1.0,
    top_k=10,
    top_p=0.9,
    eos_token_id=None,
)

print("\nBatch generated token IDs:")
print(generated_batch)

print("\nBatch generated shape:")
print(generated_batch.shape)

lengths:
tensor([1, 6, 5])

input_ids:
tensor([[59,  0,  0,  0,  0,  0],
        [ 3, 68, 77, 81, 82, 91],
        [23, 93, 59, 40, 32,  0]])

attention_mask:
tensor([[1, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 0]])

labels before shift:
tensor([[  59, -100, -100, -100, -100, -100],
        [   3,   68,   77,   81,   82,   91],
        [  23,   93,   59,   40,   32, -100]])

shift_labels used in loss:
tensor([[-100, -100, -100, -100, -100],
        [  68,   77,   81,   82,   91],
        [  93,   59,   40,   32, -100]])

logits shape:
torch.Size([3, 6, 100])

loss:
tensor(21.9101, grad_fn=<NllLossBackward0>)

Gradient exists for token embedding weight?
True

Gradient shape:
torch.Size([100, 32])

Prompt used for generation:
tensor([[59]])

Generated token IDs with top-k + top-p sampling:
tensor([[59, 59, 59, 59, 59, 59]])

Generated shape:
torch.Size([1, 6])

Generated token IDs with greedy decoding:
tensor([[59, 59, 59, 59, 59, 59]])

Greedy generated shape

In [500]:
next_token_logits = torch.rand(3, 10)
print(next_token_logits)
values, indices = torch.topk(next_token_logits, k=4, dim =-1)
min_values = values[:, -1].unsqueeze(1)
print(min_values)
next_token_logits = torch.where(next_token_logits < min_values, torch.full_like(next_token_logits, float('-inf')), next_token_logits)
print(next_token_logits)
next_token_probs = torch.softmax(next_token_logits, dim=-1)
next_token_indices = torch.multinomial(next_token_probs, num_samples=1)
print(next_token_indices)

tensor([[0.0964, 0.5031, 0.6367, 0.9438, 0.4992, 0.6234, 0.8862, 0.6286, 0.9453,
         0.4249],
        [0.9923, 0.1888, 0.8467, 0.5768, 0.4798, 0.9787, 0.7418, 0.5770, 0.3834,
         0.9275],
        [0.5312, 0.6157, 0.4669, 0.3132, 0.2334, 0.7739, 0.5899, 0.5808, 0.5241,
         0.3183]])
tensor([[0.6367],
        [0.8467],
        [0.5808]])
tensor([[  -inf,   -inf, 0.6367, 0.9438,   -inf,   -inf, 0.8862,   -inf, 0.9453,
           -inf],
        [0.9923,   -inf, 0.8467,   -inf,   -inf, 0.9787,   -inf,   -inf,   -inf,
         0.9275],
        [  -inf, 0.6157,   -inf,   -inf,   -inf, 0.7739, 0.5899, 0.5808,   -inf,
           -inf]])
tensor([[8],
        [5],
        [1]])


In [506]:
next_token_logits = torch.rand((3, 10))
sorted_values, sorted_indices = torch.sort(next_token_logits, descending=True, dim=-1)
sorted_probs = torch.softmax(sorted_values, dim=-1)
cumulative_probs = torch.cumsum(sorted_probs, dim=-1)
sorted_indices_to_remove = cumulative_probs > 0.6
sorted_indices_to_remove[:, 1:] = sorted_indices_to_remove[:, :-1].clone()
sorted_indices_to_remove[:, 0] = False

indices_to_remove = torch.zeros_like(next_token_logits, dtype=torch.bool)
indices_to_remove.scatter_(dim=-1, index=sorted_indices, src=sorted_indices_to_remove)
next_token_logits = next_token_logits.masked_fill(indices_to_remove, float('-inf'))
next_token_probs = torch.softmax(next_token_logits, dim=-1)
next_tokens = torch.multinomial(next_token_probs, num_samples=1)
print(next_tokens)


tensor([[6],
        [1],
        [8]])


In [492]:
next_token_logits = torch.rand((3, 10))
sorted_values, sorted_indices = torch.sort(next_token_logits, descending=True, dim=-1)
sorted_probs = torch.softmax(sorted_values, dim=-1)
cumulative_probs = torch.cumsum(sorted_probs, dim=-1)
sorted_indices_to_remove = cumulative_probs > 0.6
print(sorted_indices_to_remove)
sorted_indices_to_remove[:, 1:] = sorted_indices_to_remove[:, :-1].clone()
sorted_indices_to_remove[:, 0] = False
print(sorted_indices_to_remove)

indices_to_remove = torch.zeros_like(
                        next_token_logits,
                        dtype=torch.bool,
                    )

indices_to_remove.scatter_(dim=-1, index=sorted_indices, src= sorted_indices_to_remove)
print(indices_to_remove)
next_token_logits = next_token_logits.masked_fill(indices_to_remove, float('-inf'))
next_token_logits



tensor([[False, False, False, False,  True,  True,  True,  True,  True,  True],
        [False, False, False, False, False,  True,  True,  True,  True,  True],
        [False, False, False, False,  True,  True,  True,  True,  True,  True]])
tensor([[False, False, False, False, False,  True,  True,  True,  True,  True],
        [False, False, False, False, False, False,  True,  True,  True,  True],
        [False, False, False, False, False,  True,  True,  True,  True,  True]])
tensor([[ True, False,  True, False,  True, False,  True, False, False,  True],
        [False, False, False, False,  True, False,  True, False,  True,  True],
        [False, False,  True, False, False,  True,  True, False,  True,  True]])


tensor([[  -inf, 0.6405,   -inf, 0.3758,   -inf, 0.9022,   -inf, 0.4098, 0.7037,
           -inf],
        [0.5403, 0.9815, 0.6453, 0.8459,   -inf, 0.7610,   -inf, 0.5576,   -inf,
           -inf],
        [0.8057, 0.8060,   -inf, 0.9276, 0.9732,   -inf,   -inf, 0.7726,   -inf,
           -inf]])

In [444]:
next_token_logits = torch.rand((3, 10))
next_token_logits

tensor([[0.2042, 0.6451, 0.4730, 0.1312, 0.5640, 0.4397, 0.2192, 0.0923, 0.1970,
         0.9125],
        [0.2376, 0.3274, 0.3754, 0.0440, 0.9859, 0.8518, 0.2140, 0.4628, 0.5331,
         0.1631],
        [0.9997, 0.8181, 0.6298, 0.0953, 0.1133, 0.4974, 0.8741, 0.2827, 0.7437,
         0.3920]])

In [445]:
sorted_logits, sorted_indices = torch.sort(next_token_logits, descending=True, dim=-1)
print(sorted_logits)
print(sorted_indices)

tensor([[0.9125, 0.6451, 0.5640, 0.4730, 0.4397, 0.2192, 0.2042, 0.1970, 0.1312,
         0.0923],
        [0.9859, 0.8518, 0.5331, 0.4628, 0.3754, 0.3274, 0.2376, 0.2140, 0.1631,
         0.0440],
        [0.9997, 0.8741, 0.8181, 0.7437, 0.6298, 0.4974, 0.3920, 0.2827, 0.1133,
         0.0953]])
tensor([[9, 1, 4, 2, 5, 6, 0, 8, 3, 7],
        [4, 5, 8, 7, 2, 1, 0, 6, 9, 3],
        [0, 6, 1, 8, 2, 5, 9, 7, 4, 3]])


In [446]:
sorted_probs = torch.softmax(sorted_logits, dim=-1)
sorted_probs

tensor([[0.1634, 0.1251, 0.1154, 0.1053, 0.1019, 0.0817, 0.0805, 0.0799, 0.0748,
         0.0720],
        [0.1687, 0.1475, 0.1073, 0.1000, 0.0916, 0.0873, 0.0798, 0.0780, 0.0741,
         0.0658],
        [0.1507, 0.1329, 0.1257, 0.1167, 0.1041, 0.0912, 0.0821, 0.0736, 0.0621,
         0.0610]])

In [447]:
cumulative_probs = torch.cumsum(sorted_probs, dim=-1)
cumulative_probs

tensor([[0.1634, 0.2885, 0.4039, 0.5092, 0.6111, 0.6928, 0.7733, 0.8532, 0.9280,
         1.0000],
        [0.1687, 0.3162, 0.4235, 0.5234, 0.6151, 0.7024, 0.7822, 0.8601, 0.9342,
         1.0000],
        [0.1507, 0.2836, 0.4093, 0.5259, 0.6301, 0.7212, 0.8033, 0.8769, 0.9390,
         1.0000]])

In [448]:
sorted_indices_to_remove = cumulative_probs > 0.6
sorted_indices_to_remove

tensor([[False, False, False, False,  True,  True,  True,  True,  True,  True],
        [False, False, False, False,  True,  True,  True,  True,  True,  True],
        [False, False, False, False,  True,  True,  True,  True,  True,  True]])

In [449]:
sorted_indices_to_remove[:, 1:]

tensor([[False, False, False,  True,  True,  True,  True,  True,  True],
        [False, False, False,  True,  True,  True,  True,  True,  True],
        [False, False, False,  True,  True,  True,  True,  True,  True]])

In [450]:
sorted_indices_to_remove[:, 1:] = (
                        sorted_indices_to_remove[:, :-1].clone()
                    )
sorted_indices_to_remove

tensor([[False, False, False, False, False,  True,  True,  True,  True,  True],
        [False, False, False, False, False,  True,  True,  True,  True,  True],
        [False, False, False, False, False,  True,  True,  True,  True,  True]])

In [451]:
sorted_indices_to_remove[:, 0] = False
sorted_indices_to_remove

tensor([[False, False, False, False, False,  True,  True,  True,  True,  True],
        [False, False, False, False, False,  True,  True,  True,  True,  True],
        [False, False, False, False, False,  True,  True,  True,  True,  True]])

In [452]:
indices_to_remove = torch.zeros_like(
                        next_token_logits,
                        dtype=torch.bool,
                    )
indices_to_remove

tensor([[False, False, False, False, False, False, False, False, False, False],
        [False, False, False, False, False, False, False, False, False, False],
        [False, False, False, False, False, False, False, False, False, False]])

In [453]:
indices_to_remove.scatter_(
                        dim=-1,
                        index=sorted_indices,
                        src=sorted_indices_to_remove,
                    )

tensor([[ True, False, False,  True, False, False,  True,  True,  True, False],
        [ True,  True, False,  True, False, False,  True, False, False,  True],
        [False, False, False,  True,  True,  True, False,  True, False,  True]])